# Generators and Envelopes

This notebook covers the core of pysndlib: rendering sounds with `clm.Sound`, the oscil/envelope generators, and frequency modulation.

Docs:

- [with-sound](https://testcase.github.io/pysndlib/with_sound.html)
- [CLM generators](https://testcase.github.io/pysndlib/clm_generators.html)
- Original CLM documentation: [sndclm.html](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.sndlib as sndlib

SR = 44100
import pysndlib
print("pysndlib version:", pysndlib.__version__)

## The Sound context manager

All rendering happens inside `clm.Sound(...)`. At the end of the block the output file is closed and the header updated.

In [ ]:
with clm.Sound('simple_tone.wav', channels=1, srate=SR) as snd:
    osc = clm.make_oscil(440.0)          # a sine at 440 Hz
    amp = clm.make_env([0, 0, 10, 1, 90, 1, 100, 0], duration=1.0)  # attack/decay
    for i in range(SR):                  # one second of samples
        clm.outa(i, clm.env(amp) * clm.oscil(osc))

print('samples:', sndlib.mus_sound_samples('simple_tone.wav'))

In [ ]:
data, sr = clm.file2ndarray('simple_tone.wav')
ipd.Audio('simple_tone.wav', rate=SR)

In [ ]:
plt.figure(figsize=(14, 3))
plt.plot(data[0][:2000], lw=0.7)
plt.xlabel('sample'); plt.ylabel('amplitude'); plt.title('440 Hz tone with envelope')
plt.show()

## Envelopes

An envelope is a breakpoint list `[x0, y0, x1, y1, ...]` where x is in "percent of duration" (conventionally 0..100) and y is amplitude. `make_env` builds the generator, `env` reads the next value each sample.

In [ ]:
env = clm.make_env([0, 0, 25, 1, 75, 1, 100, 0], duration=0.5)
vals = [clm.env(env) for _ in range(int(SR * 0.5))]
plt.figure(figsize=(8, 2))
plt.plot(vals, lw=0.8); plt.title('envelope curve'); plt.show()

`scaler`, `offset` and `base` shape the envelope at construction time:

In [ ]:
e1 = clm.make_env([0, 0, 100, 1], duration=0.2)
e2 = clm.make_env([0, 0, 100, 1], duration=0.2, scaler=0.5, offset=0.25)
v1 = [clm.env(e1) for _ in range(200)]
v2 = [clm.env(e2) for _ in range(200)]
plt.plot(v1, label='base')
plt.plot(v2, label='scaler=0.5, offset=0.25')
plt.legend(); plt.show()

The `pysndlib.env` module has utilities for working with envelope lists:

In [ ]:
from pysndlib import env as penv

print('interleaved:', penv.interleave([0, 1], [2, 3]))
print('max y:', penv.max_envelope([0, 0, 50, 2, 100, 1]))
print('area :', penv.integrate_envelope([0, 0, 1, 1]))
print('stretch:', penv.stretch_envelope([0, 0, 100, 1], 25, 10, 75, 90))

## Frequency modulation: oscil's input

Every generator takes an input (fm for oscillators). Feeding the output of one oscil into another's input gives classic FM synthesis -- see the [FM theory notebook](06_FM_Theory.ipynb) for the mathematics.

In [ ]:
with clm.Sound('fm_tone.wav', channels=1, srate=SR):
    carrier = clm.make_oscil(440.0)
    modulator = clm.make_oscil(220.0)                  # ratio 2:1
    index = clm.make_env([0, 0, 100, 3], duration=1.0)  # index -> brightness
    for i in range(SR):
        fm = clm.env(index) * clm.hz2radians(220.0)
        clm.outa(i, clm.oscil(carrier, clm.oscil(modulator) * fm))

fm_data, _ = clm.file2ndarray('fm_tone.wav')

In [ ]:
ipd.Audio('fm_tone.wav', rate=SR)

In [ ]:
# spectra of a quiet (low index) and bright (high index) moment
def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

quiet = spectrum(fm_data[0][int(SR*0.05):int(SR*0.05)+4096])
bright = spectrum(fm_data[0][int(SR*0.85):int(SR*0.85)+4096])
f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, quiet, label='low index')
plt.plot(f, bright, alpha=0.75, label='high index')
plt.xlim(0, 4000); plt.legend(); plt.xlabel('Hz'); plt.title('FM sidebands')
plt.show()

## Other waveform generators

The CLM generator collection (all in `clm`) includes wave-table and polynomial variants -- see the [waveforms notebook](09_Waveforms.ipynb) for the full tour:

In [ ]:
gens = [('triangle', clm.make_triangle_wave(440.0)),
        ('sawtooth', clm.make_sawtooth_wave(440.0)),
        ('square  ', clm.make_square_wave(440.0)),
        ('pulse   ', clm.make_pulse_train(440.0))]
plt.figure(figsize=(14, 6))
for n, (name, g) in enumerate(gens):
    plt.subplot(4, 1, n + 1)
    plt.plot([g() for _ in range(200)], lw=0.8)
    plt.title(name.strip()); plt.ylim(-1.2, 1.2)
plt.tight_layout(); plt.show()

In [ ]:
# table_lookup plays any waveform: partials2wave builds one from harmonic/amplitude pairs
wave = clm.partials2wave([1, 1, 2, 0.5, 3, 0.25])
with clm.Sound('table_tone.wav', channels=1, srate=SR):
    t = clm.make_table_lookup(220.0, wave=wave)
    for i in range(SR // 2):
        clm.outa(i, clm.table_lookup(t))

In [ ]:
ipd.Audio('table_tone.wav', rate=SR)

## Noise

`rand` jumps between random values; `rand_interp` interpolates between them (smoother, less "zipper" noise):

In [ ]:
r1 = clm.make_rand(440.0)
r2 = clm.make_rand_interp(440.0)
a = [clm.rand(r1) for _ in range(600)]
b = [clm.rand_interp(r2) for _ in range(600)]
plt.figure(figsize=(14, 3))
plt.plot(a, lw=0.8, label='rand')
plt.plot(b, lw=0.8, label='rand_interp')
plt.legend(); plt.show()

## Summary

- `clm.Sound` frames every render; generators are made once, then called per sample
- Envelopes are breakpoint lists; `env()` advances them
- Generator inputs are how FM, vibrato and index envelopes happen

Next: [filters and DSP](03_Filters_Delay_DSP.ipynb)